In [ ]:
import pandas as pd
import arviz as az
from matplotlib import pyplot as plt
import seaborn as sns

from emu_renewal.inputs import OUTPUTS_PATH

In [ ]:
job_path = OUTPUTS_PATH / "42523356"
countries = [p.parts[-1] for p in job_path.iterdir()]
c = countries[0]
country_path = job_path / c
analyses = [a.parts[-1] for a in country_path.iterdir()]

In [ ]:
a_idatas = []
a_likes = []
for a in analyses:
    analysis_path = country_path / a
    idata = az.from_netcdf(analysis_path / "idata_filtered.nc")
    a_idatas.append(idata)
    a_likes.append(idata["sample_stats"]["lp"].to_pandas().T)
all_likes = -pd.concat(a_likes, keys=analyses, axis=1)
like_dfs = [a_idatas[i]["log_likelihood"].to_dataframe() for i in range(len(analyses))]
like_df = pd.concat(like_dfs, axis=1, keys=analyses)
for a in analyses:
    like_df.loc[:, (a, "total_ll")] = like_df[a].sum(axis=1)

In [ ]:
targets = ["total_ll"] + list(idata["log_likelihood"].data_vars)
like_component_dfs = {}
for t in targets:
    like_component_dfs[t] = like_df.xs(t, axis=1, level=1)

In [ ]:
comp_fig, axes = plt.subplots(3, 2, figsize=[11, 14])
flat_axes = axes.ravel()
for t, targ in enumerate(targets):
    ax = sns.kdeplot(like_component_dfs[targ], fill=True, ax=flat_axes[t])
    ax.set_title(targ)
    if t != 0:
        flat_axes[t].get_legend().remove()
comp_fig.suptitle(c, fontsize=15)
comp_fig.tight_layout()